# Compare Dafne + MedSAM to Ground Truth

Evaluates Dafne+MedSAM segmentations against myosegmenTUM ground truth.
Runs **fat fraction first**, then **water** — each modality writes its own CSVs.

In [1]:
import os
import re
import numpy as np
import pandas as pd
import SimpleITK as sitk
from dissector.evaluation import binary_cross_entropy, boundary_iou_3d, inter_slice_dice

In [2]:
BOUNDARY_DISTANCE = 1
GT_BASE = os.path.join('..', '..', 'myosegmenTUM')

# (muscle_name, gt_label_idx, npz_key)
MUSCLES = [
    ('R_gracilis',  5, 'Gracilis_R'),
    ('L_gracilis',  1, 'Gracilis_L'),
    ('R_sartorius', 8, 'Sartorius_R'),
    ('L_sartorius', 4, 'Sartorius_L'),
]

In [3]:
def evaluate_muscle(muscle_name, gt_label_idx, npz_key, npz_files, seg_dir, modality_tag, result_dir, csv_suffix):
    results = []
    for npz_file in npz_files:
        stem      = npz_file.replace('_dafne_medsam.npz', '')
        subject   = re.split(rf'_({modality_tag})_', stem)[0]
        m         = re.search(r'stack(\d+)', stem)
        if not m:
            print(f'  could not parse stack number: {npz_file}, skipping')
            continue
        stack_num = m.group(1)

        gt_path = os.path.join(GT_BASE, subject, 'SegmentationMasks',
                               f'combined_gt_stack{stack_num}.mha')
        if not os.path.exists(gt_path):
            print(f'  GT not found: {gt_path}, skipping')
            continue

        gt_image = sitk.ReadImage(gt_path)
        gt       = sitk.Cast(gt_image == gt_label_idx, sitk.sitkUInt8)
        gt_arr   = sitk.GetArrayFromImage(gt).astype(float)

        npz_data = np.load(os.path.join(seg_dir, npz_file))
        pred_arr = npz_data[npz_key].astype(float)

        pred_sitk = sitk.GetImageFromArray(pred_arr.astype(np.uint8))
        pred_sitk.CopyInformation(gt_image)
        pred = sitk.Cast(pred_sitk, sitk.sitkUInt8)

        dice_filter = sitk.LabelOverlapMeasuresImageFilter()
        dice_filter.Execute(gt, pred)

        if gt_arr.sum() > 0 and pred_arr.sum() > 0:
            hd_filter = sitk.HausdorffDistanceImageFilter()
            hd_filter.Execute(gt, pred)
            hd = hd_filter.GetHausdorffDistance()
        else:
            print(f'  {npz_file}: empty mask (gt={int(gt_arr.sum())} pred={int(pred_arr.sum())}), HD=NaN')
            hd = np.nan

        results.append({
            'image':                                gt_path,
            'pred_label':                           npz_file,
            f'{muscle_name}_dice':                  dice_filter.GetDiceCoefficient(),
            f'{muscle_name}_hausdorff':             hd,
            f'{muscle_name}_jaccard':               dice_filter.GetJaccardCoefficient(),
            f'{muscle_name}_volume_similarity':     dice_filter.GetVolumeSimilarity(),
            f'{muscle_name}_false_negative':        dice_filter.GetFalseNegativeError(),
            f'{muscle_name}_false_positive':        dice_filter.GetFalsePositiveError(),
            f'{muscle_name}_bce':                   binary_cross_entropy(gt_arr, pred_arr),
            f'{muscle_name}_boundary_iou_3d':       boundary_iou_3d(BOUNDARY_DISTANCE, gt_arr, pred_arr),
            f'{muscle_name}_inter_slice_dice_pred': inter_slice_dice(pred_arr),
            f'{muscle_name}_inter_slice_dice_gt':   inter_slice_dice(gt_arr),
        })

    df = pd.DataFrame(results)
    csv_path = os.path.join(result_dir, f'df_{muscle_name}_{csv_suffix}.csv')
    df.to_csv(csv_path)
    print(f'  Saved {len(df)} rows → {csv_path}')
    return df

## Fat Fraction

In [4]:
FF_SEG_DIR    = os.path.join('..', 'segmentations_fat_frac')
FF_RESULT_DIR = os.path.join('..', 'results_fat_frac')
os.makedirs(FF_RESULT_DIR, exist_ok=True)

ff_npz_files = sorted(f for f in os.listdir(FF_SEG_DIR) if f.endswith('_dafne_medsam.npz'))
print(f'Seg dir   : {os.path.abspath(FF_SEG_DIR)}')
print(f'Result dir: {os.path.abspath(FF_RESULT_DIR)}')
print(f'Found     : {len(ff_npz_files)} npz files')

Seg dir   : C:\Projects\dissector\eval_notebooks\dafne_and_medsam\segmentations_fat_frac
Result dir: C:\Projects\dissector\eval_notebooks\dafne_and_medsam\results_fat_frac
Found     : 54 npz files


In [ ]:
dfs_ff = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} ──')
    dfs_ff[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        ff_npz_files, FF_SEG_DIR, 'FATFRACTION', FF_RESULT_DIR,
        csv_suffix='dafne_medsam_fatfrac',
    )
print('\nDone.')

In [ ]:
for name, df in dfs_ff.items():
    print(f'\n── {name} ──')
    display(df[['pred_label', f'{name}_dice', f'{name}_hausdorff']].head())

## Water

In [5]:
W_SEG_DIR    = os.path.join('..', 'segs_water')
W_RESULT_DIR = os.path.join('..', 'results_water')
os.makedirs(W_RESULT_DIR, exist_ok=True)

w_npz_files = sorted(f for f in os.listdir(W_SEG_DIR) if f.endswith('_dafne_medsam.npz'))
print(f'Seg dir   : {os.path.abspath(W_SEG_DIR)}')
print(f'Result dir: {os.path.abspath(W_RESULT_DIR)}')
print(f'Found     : {len(w_npz_files)} npz files')

Seg dir   : C:\Projects\dissector\eval_notebooks\dafne_and_medsam\segs_water
Result dir: C:\Projects\dissector\eval_notebooks\dafne_and_medsam\results_water
Found     : 0 npz files


In [ ]:
dfs_w = {}
for muscle_name, gt_idx, npz_key in MUSCLES:
    print(f'\n── {muscle_name} ──')
    dfs_w[muscle_name] = evaluate_muscle(
        muscle_name, gt_idx, npz_key,
        w_npz_files, W_SEG_DIR, 'WATER', W_RESULT_DIR,
        csv_suffix='dafne_medsam_water',
    )
print('\nDone.')

In [ ]:
for name, df in dfs_w.items():
    print(f'\n── {name} ──')
    display(df[['pred_label', f'{name}_dice', f'{name}_hausdorff']].head())